# W4-T2 -- RVC voice conversion: attack CM02

Trains a small per-speaker RVC voice model for each of ~12 train-pool speakers,
then converts real speech from *other* train-pool speakers into each target's
voice -- 12 x 125 = ~1,500 clips, the CM02 target in `docs/attack_taxonomy.md`.

**Why this exists.** P-019 (`docs/problems_and_decisions.md`) measured that XTTS-v2
(CM01) compresses pitch range by ~35% relative to real speech -- it invents
prosody from text and regresses to a flat contour. That makes XTTS a legitimate
but *easy* attack. RVC starts from real speech, so the pitch contour is human and
that compression cannot happen -- it is the second, harder attack family the plan
was written around. Section 11 measures whether that prediction actually held.

**Why Kaggle, not the dev laptop.** P-017/P-018 found the RVC toolchain installs
cleanly once you are *not* on Windows fighting MSVC Build Tools. Upstream has since
dropped `fairseq` entirely, so on Linux there is nothing left to compile at all --
see **P-020**. Training and conversion both belong here.

---

## Before you run

1. **Settings -> Accelerator -> GPU T4 x2** (or P100). **Internet -> On.**
2. **+ Add Input -> your MUCS 2021 train-pool dataset.** Any layout that contains
   `mucs2021/train/{transcripts,*.wav}` works. If the upload preserves the
   `raw/mucs2021/...` structure, `src.utils.paths.resolve()` re-roots
   `data/manifests/clip_index.csv` onto it directly; if it does not, section 4
   links the corpus under a `raw/` anchor in `/kaggle/working` and uses that, so
   no re-upload is needed either way. Only the **train pool** (25 speakers) is
   used. `clip_index.csv` itself is gitignored, so section 4 rebuilds it from
   MUCS's own Kaldi tables.
3. **+ Add Input -> a small private dataset containing the signed ethics PDF**
   (`docs/ethics/mentor_signoff_2026-08-12.pdf`). It is gitignored on purpose -- it carries real signatures and this repo is
   public -- so a fresh clone never has it, and the ethics gate blocks
   generation until you supply it here.
4. The repo is **public**, so no secret is needed to *clone* it. Section 12
   *pushes* results back, and that needs **Add-ons -> Secrets -> `GITHUB_TOKEN`**
   holding a GitHub PAT with write access to this repo. Without it every other
   section still runs; only the push fails.

At the full settings below (12 targets x 100 epochs x 125 conversions) budget
roughly 6 hours end to end on a T4 -- about 3.5 h of training and 2 h of
conversion. Use the smoke values commented in section 1 to prove the chain first.

## 1. Configuration

In [ ]:
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH    = "week8-krishna-rvc-generation"

RVC_WEBUI_REPO = "https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git"

# Full CM02 run: the taxonomy's ~12 targets x 125 conversions.
N_TARGETS = 12
TRAIN_EPOCHS = 100
CONVERSIONS_PER_TARGET = 125

# Smoke values -- 1 target, few epochs -- to confirm the whole chain (preprocess ->
# f0 -> feature -> train -> convert) runs before committing a T4 session to twelve
# targets. Section 9's output is what tells you it worked.
# N_TARGETS = 1
# TRAIN_EPOCHS = 20
# CONVERSIONS_PER_TARGET = 10

## 2. Clone the repo

Public repo, no PAT needed (verified: `api.github.com/repos/...` returns 200
unauthenticated).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "codemix-deepfake-detection"


def sh(cmd, cwd=None, check=True):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd, text=True, capture_output=True)
    if r.stdout.strip():
        print(r.stdout[-3000:])
    if r.returncode != 0:
        if r.stderr.strip():
            print(r.stderr[-3000:])
        if check:
            raise SystemExit("command failed with exit %d" % r.returncode)
    return r


if REPO.exists():
    import shutil
    shutil.rmtree(REPO)
sh(["git", "clone", "--branch", BRANCH, "--depth", "1", "https://%s" % REPO_HOST, str(REPO)])
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sh("git log --oneline -3")


## 3. Ethics sign-off

The gate has no override (`src/data/ethics_gate.py`) -- this is deliberate, not a
formality. Copy the signed PDF from your Kaggle input dataset in before anything
else runs.

In [ ]:
import glob
import shutil

signoff_candidates = glob.glob("/kaggle/input/**/mentor_signoff*.pdf", recursive=True)
if not signoff_candidates:
    raise SystemExit(
        "no mentor_signoff*.pdf found under /kaggle/input -- add the ethics-signoff "
        "dataset as an Input (see 'Before you run', step 3) and re-run"
    )
Path("docs/ethics").mkdir(parents=True, exist_ok=True)
dest = Path("docs/ethics") / Path(signoff_candidates[0]).name
shutil.copy2(signoff_candidates[0], dest)
print("copied ->", dest)

from src.data.ethics_gate import signoff_status
status = signoff_status()
print(status.describe())
if not status.signed:
    raise SystemExit("ethics gate still closed -- check the copied file is not a 0-byte placeholder")


## 4. Locate the MUCS train-pool audio + resolve manifest paths

`data/manifests/clip_index.csv` is **gitignored** (`.gitignore` line 44): it holds
~14 MB of machine-local `C:\dfdata\raw\mucs2021\...` paths, so it is neither
portable nor reviewable. A fresh clone therefore never has it, and this cell
rebuilds it from MUCS's own Kaldi tables -- the same
`python -m src.data.corpora --data-root $DATA_ROOT` that `REPRODUCE.md` documents.

`src.utils.paths.resolve()` then rebases each row onto the local `DATA_ROOT` by
finding the `raw/` anchor. A Kaggle upload may not preserve that anchor and
`/kaggle/input` is read-only, so when it is missing the corpus is linked under one
in `/kaggle/working` rather than copied. Recorded as **P-020**.

In [ ]:
import pandas as pd

from src.utils.paths import resolve_column

# Find the MUCS corpus root. A Kaggle upload may or may not preserve the raw/
# anchor that src.utils.paths.resolve() rebases onto DATA_ROOT, so handle both:
# if the anchor is missing, present the corpus under one via a symlink in
# /kaggle/working (/kaggle/input is read-only).
anchored = sorted(glob.glob("/kaggle/input/**/raw/mucs2021", recursive=True))
loose = [
    p
    for p in sorted(glob.glob("/kaggle/input/**/mucs2021", recursive=True))
    if Path(p, "train", "transcripts").is_dir()
]

if anchored:
    DATA_ROOT = str(Path(anchored[0]).parent.parent)
elif loose:
    shim = WORK / "dataroot" / "raw"
    shim.mkdir(parents=True, exist_ok=True)
    link = shim / "mucs2021"
    if not link.exists():
        link.symlink_to(loose[0])
    DATA_ROOT = str(shim.parent)
    print("upload has no raw/ anchor --", link, "->", loose[0])
else:
    print("contents of /kaggle/input:")
    for p in sorted(glob.glob("/kaggle/input/*")):
        print("  ", p)
    raise SystemExit("raw/mucs2021 not found under any /kaggle/input dataset -- see step 2 above")

os.environ["DATA_ROOT"] = DATA_ROOT
print("DATA_ROOT =", DATA_ROOT)

# clip_index.csv is gitignored (.gitignore line 44 -- it holds machine-local
# absolute paths), so a fresh clone never has it. Rebuild it from the corpus's
# own Kaldi tables, exactly as REPRODUCE.md documents.
index_path = Path("data/manifests/clip_index.csv")
if not index_path.is_file():
    sh([sys.executable, "-m", "src.data.corpora", "--data-root", DATA_ROOT, "--out", str(index_path)])

index = pd.read_csv(index_path)
pools = pd.read_csv("data/manifests/speaker_pools.csv")
index = resolve_column(index, "wav_path", root=DATA_ROOT)

train_pool = set(pools.loc[pools["pool"] == "train", "speaker"].astype(str))
in_pool = index[index["speaker"].astype(str).isin(train_pool)]
print("train-pool speakers: %d, clips: %d" % (len(train_pool), len(in_pool)))
if in_pool.empty:
    raise SystemExit("no clip in the index belongs to a train-pool speaker -- check speaker_pools.csv")

# Spot-check a sample of train-pool clips actually resolve to real files.
sample = in_pool.sample(min(20, len(in_pool)), random_state=0)
missing = [p for p in sample["wav_path"] if not Path(p).is_file()]
print("spot-checked %d train-pool clips, %d missing" % (len(sample), len(missing)))
if missing:
    print("first missing path:", missing[0])
    raise SystemExit("resolved paths do not exist -- check the uploaded dataset's folder structure")

## 5. Install the RVC toolchain

Training is the RVC-WebUI repo's own scripts, not a pip package (P-017/P-018), and
conversion is that same checkout's `infer/cli.py` -- one tree, one set of weights,
nothing to version-match by hand.

**What changed since P-018 was written.** Upstream dropped `fairseq` entirely:
HuBERT is a Transformers model directory now (`infer/hubert.py`), and the training
scripts moved from `infer/modules/train/` to `train/`. So the `omegaconf`/`pip<24.1`
dance and the whole MSVC saga in P-018 no longer apply -- there is nothing left to
compile.

`requirments_*.txt` is **not** installed here on purpose. Those files pin
`numpy<2`, `pydantic<2`, `gradio 3.14` and a China PyPI mirror to keep `webui.py`'s
Gradio 3 UI working; we never launch the UI, and applying those pins to a Kaggle
image would tear out the preinstalled torch stack. The four packages below are what
the training and inference scripts actually import and Kaggle does not already ship.

In [ ]:
RVC_REPO = str(WORK / "rvc-webui")
if not Path(RVC_REPO).exists():
    sh(["git", "clone", "--depth", "1", RVC_WEBUI_REPO, RVC_REPO])
os.environ["RVC_REPO"] = RVC_REPO

# --- why the two env vars below exist -------------------------------------
# RVC's stage scripts live in subpackages (train/, infer/) but import each other
# as top-level packages: train/preprocess.py opens with "from infer.audio import
# load_audio" and "from train.dataset.slicer2 import Slicer". Run as
# `python train/preprocess.py`, CPython puts the SCRIPT's directory first on
# sys.path, which breaks that twice over:
#   1. <repo> is not on the path at all  -> ModuleNotFoundError: no module named 'infer'
#   2. <repo>/train IS first on the path -> "train" resolves to the module
#      train/train.py, not the package train/, so "from train import utils"
#      dies as a circular import of a partially initialised 'train'.
# PYTHONPATH puts the repo root on the path; PYTHONSAFEPATH (3.11+) stops the
# script directory being prepended, so train/ and infer/ resolve as the packages
# they are. Upstream's webui.py never hits this -- its packaged runtime already
# has the repo root on the path. Every stage subprocess inherits this env.
os.environ["PYTHONPATH"] = os.pathsep.join([RVC_REPO, os.environ.get("PYTHONPATH", "")]).strip(os.pathsep)
os.environ["PYTHONSAFEPATH"] = "1"

# Only what the five scripts we run import and Kaggle lacks. Everything else they
# need (torch, numpy, scipy, librosa, soundfile, transformers, tensorboard,
# scikit-learn, matplotlib) is already in the Kaggle image.
sh([sys.executable, "-m", "pip", "install", "-q",
    "av", "ffmpeg-python", "praat-parselmouth", "faiss-cpu"])

import torch
print("torch     ", torch.__version__)
print("cuda avail", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("no GPU -- set Accelerator to GPU T4 x2 and restart the session")

# Fail here rather than three stages into the first target: these are exactly the
# imports preprocess / f0 / feature / index extraction do.
import faiss, parselmouth, soundfile, transformers  # noqa: F401
import av  # noqa: F401
print("faiss / parselmouth / av / transformers import OK")

# Preflight the subprocess import path itself, so a broken sys.path surfaces here
# and not after 127 wavs have been staged.
sh([sys.executable, "-c",
    "import infer.audio, train.utils, train.dataset.slicer2; print('stage-script imports OK')"],
   cwd=RVC_REPO)

## 6. Pretrained base models

RVC-WebUI ships none of its weights in git. Four things are needed and all four are
non-optional:

- **`assets/hubert_base/`** -- the feature extractor. A *directory* (config.json,
  preprocessor_config.json, pytorch_model.bin), not the old fairseq `hubert_base.pt`.
  Downloading a single `.pt` to the old path is the failure that looks like it
  worked and then raises `FileNotFoundError` during feature extraction.
- **`assets/rmvpe/rmvpe.pt`** -- the f0 estimator.
- **`assets/pretrained_v2/f0{G,D}40k.pth`** -- the generator/discriminator pair
  training warm-starts from. Each target here has minutes, not hours, of audio --
  far below a from-scratch budget -- so the warm start is what makes a per-speaker
  model viable at all.
- **`logs/mute/`** -- the silence example every training run's `filelist.txt`
  references. Missing it fails inside the dataloader, after preprocessing.

`download_rvc_assets` fetches exactly these (~700 MB, skipping whatever is already
there) and then re-runs the preflight, so a broken download surfaces now rather
than forty minutes into a session.

In [ ]:
from src.data import rvc_generation as rvc

fetched = rvc.download_rvc_assets(RVC_REPO)
for path in fetched:
    print("fetched", path)
rvc.assert_rvc_assets(RVC_REPO)
print("\nRVC assets present and consistent")

## 7. Select RVC targets + build the conversion job table

The firewall is checked here, before any GPU time: both endpoints of every
conversion -- target *and* source -- must be train-pool speakers, no speaker is
converted into themselves, and no source clip is reused within one target. A
converted clip carries the **target's** speaker id, because the voice it holds is
the target's; that is what makes CM02's pool disjointness mean the same thing as
CM01's.

In [ ]:
config = rvc.RVCConfig(
    n_targets=N_TARGETS,
    conversions_per_target=CONVERSIONS_PER_TARGET,
    train_epochs=TRAIN_EPOCHS,
)

targets = rvc.select_target_speakers(index, pools, config)
print("%d target speaker(s) qualify (>= %.0fs each):" % (len(targets), config.min_training_seconds))
print(targets.to_string(index=False))
if targets.empty:
    raise SystemExit("no train-pool speaker has enough audio -- lower min_training_seconds or check DATA_ROOT")

jobs = rvc.build_rvc_jobs(
    index, pools, out_dir=str(WORK / "rvc_outputs"), config=config, targets=targets
)
Path("data/manifests").mkdir(parents=True, exist_ok=True)
jobs.to_csv("data/manifests/rvc_generation_jobs.csv", index=False)
print(rvc.rvc_job_summary(jobs))

## 8. Train each target's RVC voice model

Runs the 4-stage RVC-WebUI pipeline per target: preprocess -> f0 extract ->
feature extract -> train. Resumable: a target whose weight file already exists is
skipped, so a session reclaimed part-way through twelve targets continues rather
than retraining target one.

The stage scripts are launched with `PYTHONPATH`/`PYTHONSAFEPATH` set by
`rvc_generation._stage_env` -- without it `train/preprocess.py` cannot import
`infer`, and `train/train.py` shadows the `train` package. Both failures land
*after* the training wavs are staged. See **P-020**.

In [ ]:
models = {}
for target_row in targets.itertuples(index=False):
    target = str(target_row.speaker)
    clips = rvc.gather_training_clips(index, target)
    print("training %s (%d clips, %.0fs)..." % (target, len(clips), clips["duration_seconds"].sum()))
    model = rvc.train_speaker_model(
        target, clips, str(WORK / "rvc_train"), RVC_REPO, config=config
    )
    print("  ", model.describe())
    models[target] = model


## 9. Convert: real speech from other train-pool speakers -> each target's voice

In [ ]:
inferencer = rvc.load_rvc_inferencer()


def progress(i, total, row):
    print("  [%d/%d] %s -> %s" % (i, total, row["source_speaker"], row["target_speaker"]))


metadata_path = str(WORK / "rvc_outputs" / "generation_metadata.jsonl")
failures = []
written = rvc.convert_batch(
    jobs, inferencer, models, metadata_path=metadata_path, on_progress=progress, failures=failures
)
print("\nconverted %d clip(s)" % len(written))
if failures:
    print("%d failure(s) -- see the printed [FAILED] lines above" % len(failures))


## 10. Collect the artefacts

In [ ]:
import json

kept = rvc.rewrite_metadata(metadata_path)
records = rvc.read_metadata(metadata_path)
print("metadata reconciled: %d record(s) matching files on disk" % kept)
stats = rvc.rvc_generation_stats(records)
print(json.dumps(stats, indent=2))

summary = {
    "task": "W4-T2 RVC generation (CM02)",
    "n_targets": len(models),
    "models": {t: m.describe() for t, m in models.items()},
    "stats": stats,
}
(WORK / "rvc_generation_summary.json").write_text(json.dumps(summary, indent=2))
print("\nDownload from this version's Output tab:")
print("  -", WORK / "rvc_generation_summary.json")
print("  -", metadata_path)
print("  -", "data/manifests/rvc_generation_jobs.csv (copy out manually if needed)")
print("  -", WORK / "rvc_outputs", "(the converted clips themselves)")


## 11. Screen the corpus, measure its pitch, and write the artefacts

This repository cannot hold the run. The converted clips are cloned voices of
identifiable MUCS speakers in a **public** repo and `**/generated/**/*.wav` is
gitignored on purpose; the voice models are ~55 MB each and `checkpoints/**` is
ignored for the same reason. What the repo keeps is a *description* precise
enough to audit and rebuild the run: the job table, per-clip metadata with
machine-local paths rewritten portable, a mechanical quality screen, the pitch
measurement P-019's prediction turns on, and SHA-256 checksums of the weights.

`src.data.rvc_report` does the work. These cells only supply this run's numbers.

In [ ]:
import json

from src.data import generation_qa, rvc_report

CONVERTED_DIR = str(WORK / "rvc_outputs")

# A generator can return valid audio that is completely wrong and raise nothing --
# a pilot clip produced 0.83 s from a 150-character transcript, was logged as a
# success and was counted. Nobody listens to 1,500 clips, so this is mechanical.
qa_report = rvc_report.screen_rvc_jobs(jobs, CONVERTED_DIR)
Path(rvc_report.QA_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)
qa_report.to_csv(rvc_report.QA_CSV_PATH, index=False)
print("wrote", rvc_report.QA_CSV_PATH)


qa_stats = generation_qa.summarise(qa_report)
print(json.dumps(qa_stats, indent=2))
for row in qa_report[~qa_report["ok"]].head(20).itertuples(index=False):
    print("  [FAIL]", row.clip, "--", row.reason)

In [ ]:
# Pitch range -- the measurement CM02's existence rests on.
#
# P-019 predicted that RVC *cannot* compress pitch the way XTTS-v2 does, because
# voice conversion starts from a real human recording and swaps timbre only. That
# prediction is the whole justification for this attack family, so it is measured
# rather than assumed. The converted clips and the real clips they were converted
# from are measured in the same pass with the same estimator: P-019's 41.1 Hz was
# taken ad hoc with no committed code, so quoting it against a different tracker
# would compare tooling instead of attacks.
from src.data import f0_stats

pitch = rvc_report.measure_pitch(jobs, CONVERTED_DIR)
print(json.dumps(pitch, indent=2))
print()
print(pitch["verdict"])

In [ ]:
# Artefacts, with every machine-local path rewritten first.
#
# `generation_metadata.jsonl` and the job table are both written full of absolute
# /kaggle/working/... paths. Commit f60f989 gitignored the ASVspoof manifests for
# exactly that reason: an absolute path committed from one machine resolves to
# nothing on every other one. Both go through `src.utils.paths.portable` here.
records = rvc.read_metadata(metadata_path)
n_meta = rvc_report.write_portable_metadata(
    records, rvc_report.METADATA_PATH, root=DATA_ROOT
)
jobs_portable = rvc_report.write_portable_jobs(
    jobs, "data/manifests/rvc_generation_jobs.csv", root=DATA_ROOT
)

# The weights never leave Kaggle; their identity does.
model_rows = rvc_report.write_model_manifest(models)

run_summary = {
    "task": "W4-T2 RVC generation (CM02)",
    "n_targets": len(models),
    "config": {
        "train_epochs": config.train_epochs,
        "conversions_per_target": config.conversions_per_target,
        "sample_rate": config.sample_rate,
        "version": config.version,
        "f0_method": rvc._f0_method(config),
        "seed": config.seed,
    },
    "models": {t: m.describe() for t, m in models.items()},
    "stats": stats,
    "qa": qa_stats,
    "pitch": pitch,
}
Path(rvc_report.SUMMARY_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(rvc_report.SUMMARY_PATH).write_text(
    json.dumps(run_summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

print("portable metadata records:", n_meta)
print("portable job rows        :", len(jobs_portable))
print("models checksummed       :", len(model_rows))
for row in model_rows:
    print("   %s  %s  %.1f MB  %s" % (row["speaker"], row["weight_file"],
                                      row["bytes"] / 1e6, row["sha256"][:16] + "..."))

In [ ]:
# The documents. Every number below is measured in this run; nothing is a target.
import datetime
import subprocess as _sp


def _short(repo):
    return _sp.run(["git", "rev-parse", "--short", "HEAD"], cwd=repo,
                   capture_output=True, text=True).stdout.strip() or "unknown"


def _flag(needle):
    return int(qa_report["reason"].astype(str).str.contains(needle, regex=False).sum())


per_target = []
for speaker in sorted(models, key=str):
    model = models[speaker]
    group = qa_report[qa_report["speaker"].astype(str) == str(speaker)]
    per_target.append({
        "speaker": str(speaker),
        "train_clips": int(model.n_clips),
        "train_seconds": float(model.train_seconds),
        "epochs": int(model.epochs),
        "written": int((group["reason"].astype(str) != "missing").sum()),
        "qa_pass": "%d/%d" % (int(group["ok"].sum()), len(group)),
    })

context = {
    "accelerator": "%d x %s" % (torch.cuda.device_count(), torch.cuda.get_device_name(0)),
    "run_date": datetime.date.today().isoformat(),
    "branch": BRANCH,
    "commit": _short(str(REPO)),
    "rvc_commit": _short(RVC_REPO),
    "n_targets": len(models),
    "train_epochs": config.train_epochs,
    "conversions_per_target": config.conversions_per_target,
    "written": int(qa_stats.get("clips", 0)) - _flag("missing"),
    "failures": len(failures),
    "total_audio_hours": float(qa_report["duration_sec"].sum()) / 3600.0,
    "train_pool_speakers": len(train_pool),
    "sample_rate": config.sample_rate,
    "f0_method": rvc._f0_method(config),
    "pitch_floor": f0_stats.PITCH_FLOOR_HZ,
    "pitch_ceiling": f0_stats.PITCH_CEILING_HZ,
    "model_mb": (sum(r["bytes"] for r in model_rows) / max(len(model_rows), 1)) / 1e6,
    "index_rows": len(index),
    "index_speakers": int(index["speaker"].nunique()),
    "per_target": per_target,
    "qa": qa_stats,
    "pitch": pitch,
    "flag_rate": _flag("chars/s"),
    "flag_silent": _flag("near-silent"),
    "flag_clipped": _flag("clipped ("),
    "flag_short": _flag("too short"),
    "flag_missing": _flag("missing"),
}

written_docs = [
    rvc_report.write_doc(rvc_report.RESULTS_DOC_PATH, rvc_report.render_results_doc(context)),
    rvc_report.write_doc(rvc_report.QA_DOC_PATH, rvc_report.render_qa_doc(context)),
]

# The ADR log is append-only by its own rule, and the taxonomy ships targets that
# have to become measurements once a row has actually been generated. Both edits
# are idempotent, so a repeated run never leaves two P-020 rows behind.
added = rvc_report.append_decision(rvc_report.decision_entry(context))
retuned = rvc_report.update_taxonomy_row(
    "CM02",
    "**%s** *(measured)*" % format(context["written"], ","),
    str(context["n_targets"]),
)

for path in written_docs:
    print("wrote", path)
print("P-020 appended to docs/problems_and_decisions.md:", added)
print("CM02 row updated in docs/attack_taxonomy.md      :", retuned)
print()
print(pitch["verdict"])

## 12. Commit and push

Pushed with a GitHub PAT read from **Add-ons → Secrets → `GITHUB_TOKEN`**, never
written into the notebook, the git config, or any printed command: git asks for it
through `GIT_ASKPASS`, so it stays out of `argv` and out of `.git/config`.

Only the description is committed. The audio and the weights are not — they are
cloned voices of identifiable speakers in a public repo, and both are gitignored
deliberately. The cell refuses to push if any staged file still contains a
`/kaggle/` path, because a manifest full of one machine's absolute paths is junk
on every other machine (commit f60f989).

In [ ]:
import stat
import urllib.request

from kaggle_secrets import UserSecretsClient

COMMIT_PATHS = [
    "data/manifests/rvc_generation_jobs.csv",
    rvc_report.METADATA_PATH,
    rvc_report.SUMMARY_PATH,
    rvc_report.MODEL_MANIFEST_PATH,
    rvc_report.QA_CSV_PATH,
    rvc_report.QA_DOC_PATH,
    rvc_report.RESULTS_DOC_PATH,
    "docs/problems_and_decisions.md",
    "docs/attack_taxonomy.md",
]

_token = UserSecretsClient().get_secret("GITHUB_TOKEN")

# Attribute the commit to whoever owns the credential, rather than to a bot.
_who = json.loads(
    urllib.request.urlopen(
        urllib.request.Request(
            "https://api.github.com/user",
            headers={
                "Authorization": "Bearer " + _token,
                "Accept": "application/vnd.github+json",
                "User-Agent": "codemix-kaggle-run",
            },
        )
    ).read()
)
print("authenticated as", _who["login"])

# GIT_ASKPASS keeps the token out of argv and out of .git/config. The helper reads
# it from the environment, so it is never a literal in any file on disk.
_askpass = Path("/kaggle/working/.git-askpass.sh")
_askpass.write_text(
    '#!/bin/sh\ncase "$1" in\n  Username*) echo x-access-token ;;\n'
    '  Password*) echo "$GIT_PUSH_TOKEN" ;;\nesac\n',
    encoding="utf-8",
)
_askpass.chmod(_askpass.stat().st_mode | stat.S_IEXEC)
_git_env = dict(os.environ)
_git_env.update(
    GIT_ASKPASS=str(_askpass), GIT_PUSH_TOKEN=_token, GIT_TERMINAL_PROMPT="0"
)


def git(*args, check=True):
    """Run one git command in the repo. The environment is never printed."""
    print("$ git", " ".join(args))
    done = subprocess.run(["git", *args], cwd=str(REPO), env=_git_env,
                          text=True, capture_output=True)
    if done.stdout.strip():
        print(done.stdout.strip()[-4000:])
    if done.returncode != 0:
        print(done.stderr.strip()[-4000:])
        if check:
            raise SystemExit("git %s failed (%d)" % (args[0], done.returncode))
    return done


git("config", "user.name", _who["login"])
git("config", "user.email", "%d+%s@users.noreply.github.com" % (_who["id"], _who["login"]))
git("add", "--", *[p for p in COMMIT_PATHS if Path(p).exists()])

# Refuse to ship one machine's absolute paths, whatever the manifest says.
_staged = git("diff", "--cached", "--name-only").stdout.split()
_offenders = []
for _path in _staged:
    try:
        if "/kaggle/" in Path(_path).read_text(encoding="utf-8", errors="ignore"):
            _offenders.append(_path)
    except OSError:
        pass
if _offenders:
    raise SystemExit(
        "refusing to push: machine-local /kaggle/ paths in " + ", ".join(_offenders)
    )
print("\nstaged, no machine-local paths:")
git("diff", "--cached", "--stat")

_message = (
    "W4-T2: CM02 RVC generation run -- %d clips across %d targets\n\n"
    "%d target voice models trained (%d epochs, %s v2), %d clips converted, "
    "%d failures. QA pass rate %.1f%%. Median f0 IQR %.1f Hz converted vs %.1f Hz "
    "on the real source clips (%.1f%% retention), which is what P-019 predicted "
    "for voice conversion and did not get from XTTS-v2.\n\n"
    "Audio and weights are not committed (public repo, cloned voices of "
    "identifiable speakers; **/generated/**/*.wav and checkpoints/** are ignored "
    "deliberately). Checksums stand in for the weights. Manifest and metadata "
    "paths are rewritten portable.\n"
) % (
    context["written"], context["n_targets"], context["n_targets"],
    context["train_epochs"], context["sample_rate"], context["written"],
    context["failures"], qa_stats["pass_rate"],
    pitch["converted"].get("median_f0_iqr_hz", 0.0),
    pitch["real_source"].get("median_f0_iqr_hz", 0.0),
    pitch["retention_pct"],
)

if git("diff", "--cached", "--quiet", check=False).returncode == 0:
    print("\nnothing to commit -- the tree already matches this run")
else:
    git("commit", "-m", _message)
    git("push", "--dry-run", "origin", "HEAD:%s" % BRANCH)
    git("push", "origin", "HEAD:%s" % BRANCH)
    print("\npushed to %s" % BRANCH)

_askpass.unlink(missing_ok=True)
del _token, _git_env